In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
csv_file_path = '2Probe.csv'

In [ ]:
try:
    df = pd.read_csv(csv_file_path)
    print("CSV file loaded successfully!")
except FileNotFoundError:
    print(f"Error: The file at '{csv_file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

In [ ]:
df.shape

In [ ]:
df = df.iloc[1:].reset_index(drop=True)
df.shape

In [ ]:
df['row_type'].value_counts()

In [ ]:
df.rename(columns={"eit_time_s": "time_eit_s"}, inplace=True)

In [ ]:
df.head(20)

In [ ]:
df['target_force_N'].value_counts()

In [ ]:
df.loc[:, 'target_x_mm'] = df['target_x_mm'].values
df.loc[:, 'target_y_mm'] = df['target_y_mm'].values

In [ ]:
df['target_x_mm'].describe()

In [ ]:
df['target_y_mm'].describe()

In [ ]:
cols_to_exclude = ['actual_x_mm', 'actual_y_mm', 'position_time_s', 'force_time_s', 'measurement_index']
df = df.drop(columns=cols_to_exclude, errors='ignore')
df.head()

In [ ]:
df1 = df.drop(columns=['target_x_mm', 'target_y_mm', 'actual_force_N'])
df1.columns

In [ ]:
df.tail()

In [ ]:
import numpy as np

# Identify EIT columns from the current df
eit_columns = [col for col in df.columns if col.startswith('eit_')]

# Separate baseline and contact rows, ensuring their indices are reset for direct alignment
df_baseline = df[df['row_type'] == 'baseline'].reset_index(drop=True)
df_contact = df[df['row_type'] == 'contact'].reset_index(drop=True)

# Ensure both dataframes have the same number of rows for element-wise subtraction
# Since df_contact has fewer rows (4061) than df_baseline (4062), we truncate df_baseline to match.
min_rows = min(len(df_baseline), len(df_contact))
df_baseline = df_baseline.iloc[:min_rows]
df_contact = df_contact.iloc[:min_rows]

# Perform the subtraction of EIT values: contact_eit - baseline_eit
df_contact.loc[:, eit_columns] = df_contact[eit_columns].values - df_baseline[eit_columns].values

# Update df_filtered to contain only the processed contact rows and drop the 'row_type' column
# We also need to keep 'target_x_mm', 'target_y_mm', 'actual_force_N' for the localization model
df_filtered = df_contact.drop(columns=['row_type']).copy()

df_filtered['target_x_mm'] = df_filtered['target_x_mm'].round(1)
df_filtered['target_y_mm'] = df_filtered['target_y_mm'].round(1)
# Display the head of the transformed DataFrame for verification
df_filtered.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define features (X_df) and targets (y_df) from df_filtered
# EIT columns are the features
X_df = df_filtered[eit_columns]
# target_x_mm and target_y_mm are the targets
y_df = df_filtered[['target_x_mm', 'target_y_mm']]
# Keep target_force_N aligned with the data
target_force_N_df = df_filtered['target_force_N']

# Initialize StandardScaler
scaler = StandardScaler()

# Scale the EIT features
X_scaled = scaler.fit_transform(X_df)

# Convert X_scaled back to a DataFrame for easier splitting with target_force_N
X_scaled_df = pd.DataFrame(X_scaled, columns=eit_columns, index=df_filtered.index)

# Combine X_scaled_df, y_df, and target_force_N_df into a single DataFrame for splitting
combined_df = pd.concat([X_scaled_df, y_df, target_force_N_df], axis=1)

# Split combined_df into training and testing sets (80% train, 20% test)
train_combined, test_combined = train_test_split(combined_df, test_size=0.2, random_state=42)

# Extract X_train, X_test, y_train, y_test, and target_force_N for test set
X_train = train_combined[eit_columns].values
X_test = test_combined[eit_columns].values
y_train = train_combined[['target_x_mm', 'target_y_mm']]
y_test = test_combined[['target_x_mm', 'target_y_mm']]
target_force_N_test = test_combined['target_force_N']

print(f"Shape of scaled features (X_scaled): {X_scaled.shape}")
print(f"Shape of targets (y): {y_df.shape}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"target_force_N_test shape: {target_force_N_test.shape}")

In [ ]:
# Re-execute cells to ensure consistent data state

print("Reloading dataframe, re-applying column exclusions, and re-processing EIT data...")

# Cell 5jjoxK82e8A6: Reload the original DataFrame to ensure target_force_N is present
try:
    df = pd.read_csv(csv_file_path)
    print("CSV file reloaded successfully!")
except FileNotFoundError:
    print(f"Error: The file at '{csv_file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

# Cell vhPdIrxWe8zl: Drop the first row and reset index
df = df.iloc[1:].reset_index(drop=True)

df.rename(columns={"eit_time_s": "time_eit_s"}, inplace=True)

# Cell lARY1LGDe9iA: Apply column exclusions (ensure target_force_N is NOT excluded)
# This definition of cols_to_exclude ensures target_force_N is kept.
cols_to_exclude = ['actual_x_mm', 'actual_y_mm', 'position_time_s', 'force_time_s', 'measurement_index']
df = df.drop(columns=cols_to_exclude, errors='ignore')

# Cell Zh9__4t-fnt_: Perform baseline subtraction and filter data
import numpy as np

eit_columns = [col for col in df.columns if col.startswith('eit_')]

df_baseline = df[df['row_type'] == 'baseline'].reset_index(drop=True)
df_contact = df[df['row_type'] == 'contact'].reset_index(drop=True)

min_rows = min(len(df_baseline), len(df_contact))
df_baseline = df_baseline.iloc[:min_rows]
df_contact = df_contact.iloc[:min_rows]

df_contact.loc[:, eit_columns] = df_contact[eit_columns].values - df_baseline[eit_columns].values
df_filtered = df_contact.drop(columns=['row_type']).copy()

print("Dataframe preprocessing complete. Now splitting data...")

# Cell 95b8673a: Split data correctly, including target_force_N_test
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features (X_df) and targets (y_df) from df_filtered
X_df = df_filtered[eit_columns]
y_df = df_filtered[['target_x_mm', 'target_y_mm']]
target_force_N_df = df_filtered['target_force_N'] # Now target_force_N should be present

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)
X_scaled_df = pd.DataFrame(X_scaled, columns=eit_columns, index=df_filtered.index)

combined_df = pd.concat([X_scaled_df, y_df, target_force_N_df], axis=1)

train_combined, test_combined = train_test_split(combined_df, test_size=0.2, random_state=42)

X_train = train_combined[eit_columns].values
X_test = test_combined[eit_columns].values
y_train = train_combined[['target_x_mm', 'target_y_mm']]
y_test = test_combined[['target_x_mm', 'target_y_mm']]
target_force_N_test = test_combined['target_force_N']

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"target_force_N_test shape: {target_force_N_test.shape}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_df, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Now, let's define and train the ResNet model with L2 regularization. We'll use a simple ResNet-like architecture for this regression task.

In [ ]:
import torch
import numpy as np
import random

def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # if using multi-GPU
        # For deterministic algorithms:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False # Can be slower
    np.random.seed(seed)
    random.seed(seed)
    print(f"Random seed set to {seed}")

# Set a fixed random seed
SEED = 42
set_seed(SEED)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Define ResNet block for regression in PyTorch
class ResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, regularization_strength):
        super(ResNetBlock, self).__init__()
        self.dense1 = nn.Linear(in_channels, out_channels)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dense2 = nn.Linear(out_channels, out_channels)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()

        # Shortcut connection
        self.shortcut = nn.Identity()
        if in_channels != out_channels:
            self.shortcut = nn.Linear(in_channels, out_channels)

        # regularization_strength is kept for potential direct use with optimizer's weight_decay
        # or for explicit calculation outside the module if preferred.

    def forward(self, x):
        residual = x

        out = self.dense1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dense2(out)
        out = self.bn2(out)

        # L2 regularization calculation is moved to the training loop or handled by optimizer's weight_decay
        # Removed: self.add_loss(self.regularization_strength * l2_reg)

        out += self.shortcut(residual) # Add shortcut connection
        out = self.relu(out)
        return out

# Model Parameters
input_dim = X_train.shape[1] # Number of EIT channels
filters = 64 # Number of units in dense layers of blocks
regularization_strength = 0.001 # L2 regularization strength

# Build the ResNet model
class ResNetRegression(nn.Module):
    def __init__(self, input_dim, filters, regularization_strength):
        super(ResNetRegression, self).__init__()
        self.initial_dense = nn.Linear(input_dim, filters)
        self.initial_bn = nn.BatchNorm1d(filters)
        self.relu = nn.ReLU()

        # Pass regularization_strength to blocks if they need it for internal calculations
        # However, for weight_decay in optimizer, it's typically applied globally to model parameters.
        self.block1 = ResNetBlock(filters, filters, regularization_strength)
        self.block2 = ResNetBlock(filters, filters * 2, regularization_strength)
        self.block3 = ResNetBlock(filters * 2, filters * 2, regularization_strength)

        self.output_layer = nn.Linear(filters * 2, 2) # 2 outputs for x and y

        # Store for potential external use, not directly used within the model's forward pass for L2
        self.regularization_strength = regularization_strength

    def forward(self, x):
        x = self.initial_dense(x)
        x = self.initial_bn(x)
        x = self.relu(x)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.output_layer(x)
        return x

model = ResNetRegression(input_dim, filters, regularization_strength)

# Compile the model (optimizer and loss will be defined in the training loop)
# No direct 'compile' step in PyTorch, but we can print the model structure
print(model)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error # Import mean_absolute_error here

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32) # .values to get numpy array
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

# Create Validation DataLoader (using X_test, y_test for validation as requested)
val_dataset = TensorDataset(X_test_tensor, y_test_tensor)
val_loader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False) # No need to shuffle validation data

# Define loss function and optimizer
criterion = nn.MSELoss() # Mean Squared Error
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer with learning rate

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 150 # Set to 38 epochs as requested

history = {'loss': [], 'val_loss': [], 'mae': [], 'val_mae': []}

for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    train_predictions = []
    train_true_values = []

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Add L2 regularization explicitly to the loss
        if regularization_strength > 0: # Access global regularization_strength
            l2_norm = sum(p.pow(2.0).sum() for p in model.parameters() if p.requires_grad)
            loss = loss + regularization_strength * l2_norm

        loss.backward() # Backpropagate the loss
        optimizer.step() # Update model parameters

        running_loss += loss.item() * inputs.size(0)
        train_predictions.extend(outputs.detach().cpu().numpy())
        train_true_values.extend(labels.detach().cpu().numpy())

    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_mae = mean_absolute_error(np.array(train_true_values), np.array(train_predictions))

    history['loss'].append(epoch_train_loss)
    history['mae'].append(epoch_train_mae)

    # --- Validation Loop ---
    model.eval() # Set model to evaluation mode
    val_running_loss = 0.0
    val_predictions = []
    val_true_values = []
    with torch.no_grad(): # Disable gradient calculations during validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * inputs.size(0)
            val_predictions.extend(outputs.cpu().numpy())
            val_true_values.extend(labels.cpu().numpy())

    epoch_val_loss = val_running_loss / len(val_dataset)
    epoch_val_mae = mean_absolute_error(np.array(val_true_values), np.array(val_predictions))

    history['val_loss'].append(epoch_val_loss)
    history['val_mae'].append(epoch_val_mae)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {epoch_train_loss:.4f}, Train MAE: {epoch_train_mae:.4f}, Val Loss: {epoch_val_loss:.4f}, Val MAE: {epoch_val_mae:.4f}")

# Plot training history
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

# Plot MAE
plt.subplot(1, 2, 2)
plt.plot(history['mae'], label='Train MAE')
plt.plot(history['val_mae'], label='Validation MAE')
plt.title('Model Mean Absolute Error Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

Now, let's evaluate the model on the test dataset and calculate R2 score, MAE, and MSE for both x and y coordinates.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

model.eval() # Set model to evaluation mode

with torch.no_grad(): # Disable gradient calculations during evaluation
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_pred_tensor = model(X_test_tensor)
    y_pred = y_pred_tensor.cpu().numpy()

# Separate predictions and actual values for x and y
y_test_x = y_test.iloc[:, 0].values # .values to get numpy array
y_test_y = y_test.iloc[:, 1].values # .values to get numpy array

y_pred_x = y_pred[:, 0]
y_pred_y = y_pred[:, 1]

# Evaluate for target_x_mm
r2_x = r2_score(y_test_x, y_pred_x)
mae_x = mean_absolute_error(y_test_x, y_pred_x)
mse_x = mean_squared_error(y_test_x, y_pred_x)

print(f"--- Evaluation for target_x_mm ---")
print(f"R2 Score (x): {r2_x:.4f}")
print(f"MAE (x): {mae_x:.4f}")
print(f"MSE (x): {mse_x:.4f}\n")

# Evaluate for target_y_mm
r2_y = r2_score(y_test_y, y_pred_y)
mae_y = mean_absolute_error(y_test_y, y_pred_y)
mse_y = mean_squared_error(y_test_y, y_pred_y)

print(f"--- Evaluation for target_y_mm ---")
print(f"R2 Score (y): {r2_y:.4f}")
print(f"MAE (y): {mae_y:.4f}")
print(f"MSE (y): {mse_y:.4f}")

# Absolute distance error is the Euclidean point-wise distance between predicted and true (x, y)
absolute_distance_errors_mm = np.sqrt((y_pred_x - y_test_x)**2 + (y_pred_y - y_test_y)**2)
mae_absolute_distance_mm = np.mean(absolute_distance_errors_mm)

print(f"\n--- Absolute Distance Error ---")
print(f"MAE absolute distance: {mae_absolute_distance_mm:.4f} mm")


In [ ]:
model_save_path = 'resnet_regression_model_2_probe.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

In [ ]:
import pandas as pd
import numpy as np

# Ensure y_test is a numpy array for element-wise operations
# Assuming y_test and y_pred are available from the previous evaluation cell
y_test_np = y_test.values if isinstance(y_test, pd.DataFrame) else y_test

# Calculate squared errors for each coordinate for each sample
squared_errors = (y_pred - y_test_np)**2

# Calculate the sum of squared errors across x and y for total loss contribution per sample
sample_loss = np.sum(squared_errors, axis=1)

# Calculate the absolute Euclidean distance error in mm for each predicted (x, y) point
absolute_distance_error_mm = np.sqrt(sample_loss)

# Add sample_loss to the test_combined DataFrame
# Ensure the index is aligned, as test_combined already has the correct indices after train_test_split
test_set_with_loss = test_combined.copy()
test_set_with_loss['sample_loss'] = sample_loss
test_set_with_loss['absolute_distance_error_mm'] = absolute_distance_error_mm

print(f"MAE absolute distance: {absolute_distance_error_mm.mean():.4f} mm")

# Sort the DataFrame by 'sample_loss' in descending order
test_set_with_loss_sorted = test_set_with_loss.sort_values(by='sample_loss', ascending=False)

test_set_with_loss_sorted.head(10)

In [ ]:
index_outliers_test = test_set_with_loss_sorted[test_set_with_loss_sorted['sample_loss']> 30].index
index_outliers_test.shape

### Re-evaluating Model Performance After Removing Outliers

Now, I will remove the identified outlier data points from the test set (`X_test` and `y_test`) and then re-evaluate the model's performance. This will show how the model performs on the 'cleaner' data.

In [ ]:
# Drop the identified outliers from the test_combined DataFrame
test_set_without_outliers = test_combined.drop(index_outliers_test)

# Re-extract X_test and y_test from the filtered DataFrame
X_test_new = test_set_without_outliers[eit_columns].values
y_test_new = test_set_without_outliers[['target_x_mm', 'target_y_mm']].values

print(f"Original X_test shape: {X_test.shape}")
print(f"New X_test shape (without outliers): {X_test_new.shape}")
print(f"Original y_test shape: {y_test.shape}")
print(f"New y_test shape (without outliers): {y_test_new.shape}")

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

model.eval() # Set model to evaluation mode

with torch.no_grad(): # Disable gradient calculations during evaluation
    X_test_new_tensor = torch.tensor(X_test_new, dtype=torch.float32).to(device)
    y_pred_new_tensor = model(X_test_new_tensor)
    y_pred_new = y_pred_new_tensor.cpu().numpy()

# Separate predictions and actual values for x and y for the new test set
y_test_new_x = y_test_new[:, 0]
y_test_new_y = y_test_new[:, 1]

y_pred_new_x = y_pred_new[:, 0]
y_pred_new_y = y_pred_new[:, 1]

# Evaluate for target_x_mm on the new test set
r2_x_new = r2_score(y_test_new_x, y_pred_new_x)
mae_x_new = mean_absolute_error(y_test_new_x, y_pred_new_x)
mse_x_new = mean_squared_error(y_test_new_x, y_pred_new_x)

print(f"--- Evaluation for target_x_mm (without outliers) ---")
print(f"R2 Score (x): {r2_x_new:.4f}")
print(f"MAE (x): {mae_x_new:.4f}")
print(f"MSE (x): {mse_x_new:.4f}\n")

# Evaluate for target_y_mm on the new test set
r2_y_new = r2_score(y_test_new_y, y_pred_new_y)
mae_y_new = mean_absolute_error(y_test_new_y, y_pred_new_y)
mse_y_new = mean_squared_error(y_test_new_y, y_pred_new_y)

print(f"--- Evaluation for target_y_mm (without outliers) ---")
print(f"R2 Score (y): {r2_y_new:.4f}")
print(f"MAE (y): {mae_y_new:.4f}")
print(f"MSE (y): {mse_y_new:.4f}")

# Absolute distance error is the Euclidean point-wise distance between predicted and true (x, y)
absolute_distance_errors_new_mm = np.sqrt((y_pred_new_x - y_test_new_x)**2 + (y_pred_new_y - y_test_new_y)**2)
mae_absolute_distance_new_mm = np.mean(absolute_distance_errors_new_mm)

print(f"\n--- Absolute Distance Error (without outliers) ---")
print(f"MAE absolute distance: {mae_absolute_distance_new_mm:.4f} mm")

In [ ]:
model_save_path = 'resnet_regression_model_2_probe_no_outliers.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

In [ ]:
df_errors = df_filtered.iloc[index_outliers_test]
df_errors.head()

In [ ]:
actual_x = df_errors['target_x_mm']
actual_y = df_errors['target_y_mm']

# Calculate Euclidean distance error for (x,y) coordinates
distance_errors = test_set_with_loss_sorted.loc[index_outliers_test, 'absolute_distance_error_mm'].values

In [ ]:
plt.figure(figsize=(10, 8))
scatter = plt.scatter(actual_x, actual_y, c=distance_errors, cmap='viridis', s=50, alpha=0.7)
plt.colorbar(scatter, label='Euclidean Distance Error (mm)')
plt.title('Heatmap of Euclidean Distance Errors for (target_x_mm, target_y_mm)')
plt.xlabel('Actual target_x_mm')
plt.ylabel('Actual target_y_mm')
plt.gca().set_aspect('equal', adjustable='box') # Set aspect ratio to auto
plt.grid(True)
plt.show()

### Visualizing EIT Profiles for Outliers and Location Comparisons

Below are plots for each of the 99 identified outliers. For each outlier:

*   **Left Plot:** Shows the EIT channel values (0-207) for the specific outlier data point.
*   **Right Plot:** Shows EIT channel values for all data points in `df_filtered` that have the same `target_x_mm` and `target_y_mm` as the outlier. The lines are colored by `target_force_N` to highlight variations at that location.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure eit_columns are correctly identified
eit_columns = [col for col in df_filtered.columns if col.startswith('eit_')]

# Iterate through each outlier in df_errors
for i, (original_idx, outlier_row) in enumerate(df_errors.iterrows()):
    target_x = outlier_row['target_x_mm']
    target_y = outlier_row['target_y_mm']
    outlier_eit_values = outlier_row[eit_columns].values
    outlier_force = outlier_row['target_force_N']

    # Find all rows in df_filtered that match the target_x and target_y of the outlier
    matching_rows_filtered = df_filtered[
        (df_filtered['target_x_mm'] == target_x) &
        (df_filtered['target_y_mm'] == target_y)
    ]

    if not matching_rows_filtered.empty:
        # Calculate global min and max for EIT values across both outlier and matching rows
        all_eit_values = np.concatenate([outlier_eit_values] + [row[eit_columns].values for _, row in matching_rows_filtered.iterrows()])
        global_eit_min = all_eit_values.min()
        global_eit_max = all_eit_values.max()

        fig, axes = plt.subplots(1, 2, figsize=(20, 6))
        fig.suptitle(f"Outlier {i+1} (Original Index: {original_idx}) at (x={target_x}, y={target_y})", fontsize=16)

        # Plot 1: EIT profile for the current outlier
        axes[0].plot(range(len(eit_columns)), outlier_eit_values, color='red', linewidth=2)
        axes[0].set_title(f'EIT Profile for the Outlier (Force: {outlier_force:.1f}N)')
        axes[0].set_xlabel('EIT Channel')
        axes[0].set_ylabel('EIT Value')
        axes[0].set_ylim(global_eit_min, global_eit_max) # Set global y-limit
        axes[0].grid(True)

        # Plot 2: EIT profiles including the outlier and up to 4 other matching samples
        # Plot the outlier first in the right plot for direct comparison
        axes[1].plot(range(len(eit_columns)), outlier_eit_values, color='red', linewidth=2, label=f'Outlier (Force: {outlier_force:.1f}N)')

        # Get other matching rows, excluding the current outlier by its original index
        other_matching_rows = matching_rows_filtered[matching_rows_filtered.index != original_idx]

        # Iterate through up to 4 additional matching rows
        for j, (match_idx, match_row) in enumerate(other_matching_rows.head(4).iterrows()):
            match_eit_values = match_row[eit_columns].values
            match_force = match_row['target_force_N']
            axes[1].plot(range(len(eit_columns)), match_eit_values, alpha=0.7, label=f'Sample {j+1} (Force: {match_force:.1f}N)')

        axes[1].set_title(f'EIT Profiles at (x={target_x}, y={target_y}) from df_filtered (Outlier vs. 4 samples)')
        axes[1].set_xlabel('EIT Channel')
        axes[1].set_ylabel('EIT Value')
        axes[1].set_ylim(global_eit_min, global_eit_max) # Set global y-limit
        axes[1].grid(True)
        axes[1].legend(title='Sample Type') # Add legend to distinguish outlier

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
    else:
        print(f"No matching location found in df_filtered for outlier at (x={target_x}, y={target_y})")

In [ ]:
df.iloc[8600, :]

In [ ]:
import matplotlib.pyplot as plt

# Extract time_eit_s from df_errors
outlier_times = df_errors['time_eit_s']

plt.figure(figsize=(15, 3))
# Plot points horizontally, assigning a constant y-value
plt.scatter(outlier_times, [0] * len(outlier_times), marker='o', s=50, color='blue', alpha=0.7)

# Set x-axis range as requested
plt.xlim(0, 218360)

plt.title('Distribution of Outlier time_eit_s across Total Time Range')
plt.xlabel('time_eit_s')
plt.yticks([]) # Remove y-axis ticks as they are not meaningful for a constant value
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a sequential index for the test set data for plotting purposes
sequential_indices = np.arange(len(y_test)) # This will go from 0 to len(y_test)-1

# Determine chunk size
chunk_size = 100

# Iterate through the data in chunks using the sequential_indices
for i in range(0, len(sequential_indices), chunk_size):
    # Get the slice for the current chunk from the sequential_indices
    current_sequential_indices = sequential_indices[i : i + chunk_size]

    # Get the corresponding actual and predicted values for the current chunk
    # These slices need to be based on the sequential position, not original index
    current_y_test_x = y_test_x[current_sequential_indices]
    current_y_pred_x = y_pred_x[current_sequential_indices]
    current_y_test_y = y_test_y[current_sequential_indices]
    current_y_pred_y = y_pred_y[current_sequential_indices]

    plt.figure(figsize=(15, 6))

    # Plot for x-coordinates for the current chunk
    # The x-axis should be current_sequential_indices
    plt.subplot(1, 2, 1)
    plt.scatter(current_sequential_indices, current_y_test_x, alpha=0.6, label='Actual X', s=20)
    plt.scatter(current_sequential_indices, current_y_pred_x, alpha=0.6, label='Predicted X', s=20)
    plt.title(f'Actual vs. Predicted X-coordinates (Test Samples {current_sequential_indices.min()}-{current_sequential_indices.max()})')
    plt.xlabel('Test Sample Index (Sequential)')
    plt.ylabel('target_x_mm')
    plt.legend()
    plt.grid(True)

    # Plot for y-coordinates for the current chunk
    # The x-axis should be current_sequential_indices
    plt.subplot(1, 2, 2)
    plt.scatter(current_sequential_indices, current_y_test_y, alpha=0.6, label='Actual Y', s=20)
    plt.scatter(current_sequential_indices, current_y_pred_y, alpha=0.6, label='Predicted Y', s=20)
    plt.title(f'Actual vs. Predicted Y-coordinates (Test Samples {current_sequential_indices.min()}-{current_sequential_indices.max()})')
    plt.xlabel('Test Sample Index (Sequential)')
    plt.ylabel('target_y_mm')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()